In [ ]:
import torch

# Example PINN model
class SimplePINN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = torch.nn.Linear(1, 10)
        self.fc2 = torch.nn.Linear(10, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        return self.fc2(x)

model = SimplePINN()

# Example input data (physics residual points)
x = torch.linspace(0, 1, 100).unsqueeze(-1).requires_grad_(True)

# Example physics residual function (e.g., PDE residual)
def residuals(model, x):
    u_pred = model(x)
    du_dx = torch.autograd.grad(u_pred, x, torch.ones_like(u_pred), create_graph=True)[0]
    residual = du_dx - torch.cos(x)  # Example PDE: du/dx = cos(x)
    return residual.flatten()

# LM hyperparameters
lmbda = 1e-3  # initial damping factor
max_iter = 20

optimizer = None  # LM doesn't use built-in optimizer

for iter in range(max_iter):
    model.zero_grad()

    # Compute residual
    res = residuals(model, x)  # shape [N]
    loss = 0.5 * torch.sum(res**2)

    # Compute Jacobian explicitly using autograd
    num_params = sum(p.numel() for p in model.parameters())
    J = torch.zeros(len(res), num_params)

    for i, r in enumerate(res):
        grad_r = torch.autograd.grad(r, model.parameters(), retain_graph=True)
        J[i, :] = torch.cat([g.flatten() for g in grad_r])

    # LM step (solving linear system)
    A = J.T @ J + lmbda * torch.eye(num_params)
    g = J.T @ res.detach()

    # Solve for parameter update Δθ
    delta, _ = torch.solve(-g.unsqueeze(1), A)
    delta = delta.squeeze()

    # Update parameters manually
    with torch.no_grad():
        idx = 0
        for p in model.parameters():
            param_num = p.numel()
            p += delta[idx:idx+param_num].view_as(p)
            idx += param_num

    # Optional: adapt λ based on improvement (simple example)
    new_res = residuals(model, x)
    new_loss = 0.5 * torch.sum(new_res**2)

    if new_loss < loss:
        lmbda *= 0.7  # decrease λ if improved
    else:
        lmbda *= 2.0  # increase λ if not improved

    print(f"Iter {iter}, Loss: {new_loss.item():.6f}, Lambda: {lmbda:.6f}")


## PINN

In [48]:
# Define the physics-based constraints (example: 1D heat conduction equation)
def physics_loss(model, x_phys):
    u_pred = model(x_phys)
    # Example physics-based constraint: Laplace(u) = 0
    u_x = tf.gradients(u_pred, x_phys)[0]
    u_xx = tf.gradients(u_x, x_phys)[0]
    physics_loss = tf.reduce_mean(tf.square(u_xx))
    return physics_loss

# Define the data-driven loss
def data_loss(model, x_data, y_data):
    y_pred = model(x_data)
    data_loss = tf.reduce_mean(tf.square(y_pred - y_data))
    return data_loss

# Define the PINN model
class PINNModel(keras.Model):
    def __init__(self):
        super(PINNModel, self).__init__()
        self.hidden1 = layers.Dense(50, activation='tanh', input_dim=1)
        self.hidden2 = layers.Dense(50, activation='tanh')
        self.output_layer = layers.Dense(1, activation='linear')

    def call(self, x):
        x = self.hidden1(x)
        x = self.hidden2(x)
        return self.output_layer(x)

# Generate training data (example: 1D heat conduction)
x_data = np.random.uniform(-1, 1, (100, 1))  # Random training points
y_data = np.sin(np.pi * x_data)  # Example function

# Generate physics-based training data
x_phys = np.linspace(-1, 1, 100).reshape(-1, 1)

# Create PINN model
model = PINNModel()

# Define optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.001)

# Training loop
epochs = 10000
for epoch in range(epochs):
    # Compute total loss
    with tf.GradientTape(persistent=True) as tape:
        physics_loss_value = physics_loss(model, x_phys)
        data_loss_value = data_loss(model, x_data, y_data)
        total_loss = data_loss_value + 1.0 * physics_loss_value  # Adjust the weight for physics_loss

    # Compute gradients and update model parameters
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # Print loss every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Total Loss: {total_loss.numpy()}, Data Loss: {data_loss_value.numpy()}, Physics Loss: {physics_loss_value.numpy()}")

# After training, use the model for predictions
x_test = np.linspace(-1, 1, 100).reshape(-1, 1)
y_pred = model(x_test)

# Plot results
import matplotlib.pyplot as plt

plt.scatter(x_data, y_data, label='Training Data')
plt.plot(x_test, y_pred, label='PINN Prediction', color='r')
plt.xlabel('Input')
plt.ylabel('Output')
plt.legend()
plt.show()


AttributeError: 'NoneType' object has no attribute 'op'

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

C:\Programs\Python39\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import deepxde as dde

Using backend: tensorflow.compat.v1
Other supported backends: tensorflow, pytorch, jax, paddle.
paddle supports more examples now and is recommended.



Instructions for updating:
non-resource variables are not supported in the long term


In [37]:
geom = dde.geometry.geometry_2d.Rectangle([-1,-1], [1,1])
timedomain = dde.geometry.TimeDomain(0,1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

def pde(X, u):
    du_X = tf.gradients(u, X)[0]
    du_x, du_y, du_t = du_X[:, 0:1], du_X[:, 1:2], du_X[: 2:3]
    du_xx = tf.gradients(du_x, X)[0][:, 0:1]
    du_yy = tf.gradients(du_y, X)[0][:, 1:2]
    
    return (du_t - 0.5*(du_xx + du_yy))

In [38]:
def func(x):
    return np.sin(np.pi * x[:, 0:1]) * np.exp(-x[:, 1:2]) * np.exp(-x[:, 2:3])

bc = dde.DirichletBC(geomtime, func, lambda _, on_boundary: on_boundary)
ic = dde.IC(geomtime, func, lambda _, on_initial: on_initial)

data = dde.data.TimePDE(
    geomtime,
    pde,
    [bc, ic],
    num_domain=4000,
    num_boundary=2000,
    solution=func,
    num_test=1000,
)

In [39]:
layer_size = [3] + [32]*4 + [1]
activation = "tanh"
initializer = "Glorot uniform"
optimizer = "adam"

net = dde.maps.FNN(layer_size, activation, initializer)

In [40]:
model = dde.Model(data, net)
model.compile(optimizer, lr=0.001)

Compiling model...
Building feed-forward neural network...
'build' took 0.037583 s

'compile' took 0.388530 s



In [41]:
losshistory, train_state = model.train(iterations=10)

Training model...

Step      Train loss                        Test loss                         Test metric
0         [6.47e-02, 4.14e-01, nan]         [6.54e-02, 4.14e-01, nan]         []  

Best model at step 0:
  train loss: inf
  test loss: inf
  test metric: 

'train' took 0.477652 s



In [43]:
x_data = np.linspace(-1,1,num=100)
y_data = np.linspace(-1,1,num=100)
t_data = np.linspace(0,1,num=21)
test_x, test_t, test_y = np.meshgrid(x_data, t_data, y_data)
test_domain = np.vstack((np.ravel(test_x), np.ravel(test_y), np.ravel(test_t))).T
predicted_solution = model.predict(test_domain)
residual = model.predict(test_domain, operator=pde)

In [46]:
predicted_solution

array([[-0.16900073],
       [-0.16882811],
       [-0.16864628],
       ...,
       [-0.05108402],
       [-0.04993356],
       [-0.04877887]], dtype=float32)